In [1]:
import tensorflow as tf 
from tensorflow.keras import layers,models 

In [2]:
#LeNet-Like Model (simple Start)

model = models.Sequential([
    layers.Conv2D(6,(5,5),activation='relu',input_shape=(32,32,1)),
    layers.AveragePooling2D(pool_size=(2,2)),

    layers.Conv2D(16,(5,5),activation='relu'),
    layers.AveragePooling2D(pool_size=(2,2)),

    layers.Flatten(),
    layers.Dense(120,activation='relu'),
    layers.Dense(84,activation='relu'),
    layers.Dense(10,activation='softmax')
])

d:\BROTO\week28\dlenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
#AlexNet-like Model

model = models.Sequential([
    layers.Conv2D(96,(11,11),strides=4,activation='relu',input_shape=(224,224,3)),
    layers.MaxPooling2D(pool_size=(3,3),strides=2),

    layers.Conv2D(256,(5,5),activation='relu',padding='same'),
    layers.MaxPooling2D(pool_size=(3,3),strides=2),

    layers.Conv2D(384,(3,3),activation='relu',padding='same'),
    layers.Conv2D(384,(3,3),activation='relu',padding='same'),
    layers.Conv2D(256,(3,3),activation='relu',padding='same'),
    layers.MaxPooling2D(pool_size=(3,3),strides=2),

    layers.Flatten(),
    layers.Dense(4096,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4096,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1000,activation='softmax')
])

In [4]:
#VGG-like Model

def vgg_block(filters):
    return models.Sequential([
        layers.Conv2D(filters,(3,3),activation='relu',padding='same'),
        layers.Conv2D(filters,(3,3),activation='relu',padding='same'),
        layers.MaxPooling2D((2,2))
    ])

model=models.Sequential([
    vgg_block(64),
    vgg_block(128),
    vgg_block(256),

    layers.Flatten(),
    layers.Dense(4096,activation='relu'),
    layers.Dense(10,activation='softmax')
])

In [5]:
#ResNet Block (important)

def res_block(x,filters):
    shortcut =x
    x=layers.Conv2D(filters,(3,3),padding='same',activation='relu')(x)
    x=layers.Conv2D(filters,(3,3),padding='same')(x)

    x=layers.Add()([x,shortcut]) #This is the skip connection
    x=layers.Activation('relu')(x)

    return x

## Using Prebuilt Models (Real Practice)

In [6]:
from tensorflow.keras.applications import ResNet50

model = ResNet50(weights='imagenet')

## Import and Comparing Each Model 

In [7]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

#load Data 
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0


d:\BROTO\week28\dlenv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [8]:
#Resize
x_train = tf.image.resize(x_train[:10000], (96,96))
y_train = y_train[:10000]

x_test = tf.image.resize(x_test[:2000], (96,96))
y_test = y_test[:2000]

In [9]:
#function for model
def create_model(base):
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    output = layers.Dense(10, activation='softmax')(x)

    model = models.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [10]:
#load and train model VGG16
from tensorflow.keras.applications import VGG16

vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(96,96,3))
vgg_model = create_model(vgg_base)

vgg_model.fit(x_train, y_train, epochs=2, batch_size=32)

Epoch 1/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 896s 3s/step - accuracy: 0.1975 - loss: 2.1669
Epoch 2/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 817s 3s/step - accuracy: 0.2803 - loss: 1.8744


In [11]:
from tensorflow.keras.applications import ResNet50

resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(96,96,3))
resnet_model = create_model(resnet_base)

resnet_model.fit(x_train, y_train, epochs=2, batch_size=32)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 17s 0us/step
Epoch 1/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 620s 2s/step - accuracy: 0.5590 - loss: 1.2972
Epoch 2/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 573s 2s/step - accuracy: 0.6423 - loss: 1.0465


In [12]:
from tensorflow.keras.applications import InceptionV3

inception_base = InceptionV3(weights='imagenet', include_top=False, input_shape=(96,96,3))
inception_model = create_model(inception_base)

inception_model.fit(x_train, y_train, epochs=2, batch_size=32)

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 16s 0us/step
Epoch 1/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 290s 781ms/step - accuracy: 0.4683 - loss: 1.5505
Epoch 2/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 239s 763ms/step - accuracy: 0.5533 - loss: 1.3093


In [13]:
print("VGG16:", vgg_model.evaluate(x_test, y_test)[1])
print("ResNet50:", resnet_model.evaluate(x_test, y_test)[1])
print("InceptionV3:", inception_model.evaluate(x_test, y_test)[1])

63/63 ━━━━━━━━━━━━━━━━━━━━ 56s 878ms/step - accuracy: 0.3145 - loss: 1.7839
VGG16: 0.31450000405311584
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 391ms/step - accuracy: 0.0995 - loss: 4.7179
ResNet50: 0.09950000047683716
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 176ms/step - accuracy: 0.4990 - loss: 1.6571
InceptionV3: 0.49900001287460327


In [ ]:
#LeNet-5(From Scratch)
import tensorflow as tf 
from tensorflow.keras import layers,models

#Load Data 
(X_train,y_train),(X_test,y_test)=tf.keras.datasets.mnist.load_data()

#Normalize
X_train=X_train/255.0
X_test=X_test/255.0

#Channel Dimension
X_train=X_train[...,tf.newaxis]
X_test=X_test[...,tf.newaxis]

#resize
X_train=tf.image.resize(X_train,(32,32))
X_test=tf.image.resize(X_test,(32,32))


model = models.Sequential([
    layers.Conv2D(6,(5,5),activation='relu',input_shape=(32,32,1)),
    layers.AveragePooling2D(pool_size=(2,2)),

    layers.Conv2D(16,(5,5),activation='relu'),
    layers.AveragePooling2D(pool_size=(2,2)),

    layers.Flatten(),
    layers.Dense(120,activation='relu'),
    layers.Dense(84,activation='relu'),
    layers.Dense(10,activation='softmax')

])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics =['accuracy']
)

model.summary()

#model training 
model.fit(X_train,y_train,epochs=2,validation_data=(X_test,y_test))

#Used for: Educational purposes, MNIST-level problems

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_14 (Conv2D)              │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_12            │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_13            │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9369 - loss: 0.2034 - val_accuracy: 0.9716 - val_loss: 0.0934
Epoch 2/2
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9796 - loss: 0.0663 - val_accuracy: 0.9780 - val_loss: 0.0682


In [12]:
#AlexNet (Simplified Version)

model = models.Sequential([
    layers.Conv2D(96,(11,11),strides=4,activation='relu',input_shape=(224,224,3)),
    layers.MaxPooling2D(3,strides=2),

    layers.Conv2D(256,(5,5),padding='same',activation='relu'),
    layers.MaxPooling2D(3,strides=2),

    layers.Conv2D(384,(3,3),padding='same',activation='relu'),
    layers.Conv2D(384,(3,3),padding='same',activation='relu'),
    layers.Conv2D(256,(3,3),padding='same',activation='relu'),
    layers.MaxPooling2D(3,strides=2),

    layers.Flatten(),
    layers.Dense(4096,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4096,activation='relu'),
    layers.Dense(10,activation='softmax')    
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


d:\BROTO\week28\dlenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [1]:
#Advanced Learning
import tensorflow as tf 
from tensorflow.keras import layers,models
from tensorflow.keras.applications import VGG16


#load Data 
(X_train,y_train,),(X_test,y_test)=tf.keras.datasets.cifar10.load_data()

#normalize
X_train=X_train/255.0
X_test=X_test/255.0

#resize
X_train=tf.image.resize(X_train,(32,32))
X_test=tf.image.resize(X_test,(32,32))

#base model
base_model=VGG16(
    input_shape=(32,32,3),
    include_top=False,
    weights='imagenet'
)

#freezing
base_model.trainable=False

#model
model=models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(224,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='softmax')
])

#model compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#model training
model.fit(X_train,y_train,epochs=2,validation_split=0.5)
#model evaluation
model.evaluate(X_test,y_test)

d:\BROTO\week28\dlenv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 294s 373ms/step - accuracy: 0.4319 - loss: 1.6138 - val_accuracy: 0.5301 - val_loss: 1.3492
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 289s 330ms/step - accuracy: 0.5221 - loss: 1.3744 - val_accuracy: 0.5557 - val_loss: 1.2663
313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 169ms/step - accuracy: 0.5523 - loss: 1.2813


[1.2812837362289429, 0.552299976348877]

In [ ]:
#ResNet 
import tensorflow as tf 
from tensorflow.keras import layers,Dense
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.application import ResNet50

#load Data 
(X_train,y_train),(X_test,y_test)=tf.keras.datasets.cifar10.load_data()

#normalize
X_train=X_train/255.0
X_test=X_test/255.0

#resize
X_train=tf.image.resize(X_train(32,32))
X_test=tf.image.resize(X_test(32,32))

#base model
base_model=ResNet50(
    input_shape=(32,32,3),
    include_top=False,
    weights='imagenet'
)
#freezing 
base_model.trainable=False

#model
model=models.Sequential([
    base_model,
    layers.Flatten(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256,activation='relu'),
    layers.Dense(10,activation='softmax')
])

#model training 
model.fit(X_train,y_train,epochs=2,validation_split=0.1)

#Model evaluate
model.evaluate(X_test,y_test)
